# Network analysis of learned AKOrN oscillators

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from pathlib import Path
import einops
from einops import rearrange
from sklearn.decomposition import PCA
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Add source directory to path
#sys.path.append('/source')
from source.models.classification.my_knet import MyAKOrN

from source.models.classification.analysis_utils import AKOrNStaticAnalyzer

from source.data.augs import augmentation_strong

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Define the sweep directories
results_dir = Path("results")
sweep_dirs = [
    "sweep_20250708_581384.opbs_0",
    "sweep_20250708_581385.opbs_1",
    "sweep_20250708_581386.opbs_2",
    "sweep_20250708_581387.opbs_3",
    "sweep_20250708_581388.opbs_4",
    "sweep_20250708_581389.opbs_5",
    "sweep_20250708_581390.opbs_6",
    "sweep_20250708_581391.opbs_7",
    "sweep_20250708_581392.opbs_8",
    "sweep_20250708_581393.opbs_9",
    "sweep_20250708_581394.opbs_10",
    "sweep_20250708_581395.opbs_11",
    "sweep_20250708_581396.opbs_12",
    "sweep_20250708_581397.opbs_13",
    "sweep_20250708_581398.opbs_14",
    "sweep_20250708_581399.opbs_15",
    "sweep_20250708_581400.opbs_16",
    "sweep_20250708_581401.opbs_17"
]

print(f"Found {len(sweep_dirs)} sweep directories")

## 1. Load Learned Model and Configuration

In [ ]:
# Load the best model checkpoint
checkpoint_path = "../results/20250704_570979.opbs/my_akorn_cifar10_final.pth"
config_path = "../results/20250704_570979.opbs/parameters.json"

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
if 'epoch' in checkpoint_path:
    print(f"\nLoaded checkpoint from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
elif 'final' in checkpoint_path:
    print(f"\nLoaded final checkpoint with accuracy {checkpoint['final_accuracy']:.2f}%")

# Create model with same configuration
model =MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Import network analysis utilities
from source.kuramoto_network_metrics import (
    compute_all_metrics, 
    spectral_metrics, 
    strength_metrics,
    community_metrics,
    path_metrics,
    betweenness_metrics,
    graph_from_K
)
import networkx as nx

print("Network analysis utilities imported successfully!")

In [ ]:
## 2. Extract Connectivity Matrices from Learned Model

def extract_coupling_matrices(model, layer_idx=None):
    """Extract coupling matrices from AKOrN model layers."""
    coupling_matrices = {}
    
    if layer_idx is not None:
        # Extract from specific layer
        layer = model.layers[layer_idx]
        if hasattr(layer[2], 'connectivity'):
            weight = layer[2].connectivity.weight.detach().cpu().numpy()
            coupling_matrices[layer_idx] = weight
            print(f"Layer {layer_idx}: Extracted coupling matrix shape {weight.shape}")
    else:
        # Extract from all layers
        for i, layer in enumerate(model.layers):
            if hasattr(layer[2], 'connectivity'):
                weight = layer[2].connectivity.weight.detach().cpu().numpy()
                coupling_matrices[i] = weight
                print(f"Layer {i}: Extracted coupling matrix shape {weight.shape}")
    
    return coupling_matrices

# Extract coupling matrices from all layers
coupling_matrices = extract_coupling_matrices(model)

print(f"\nExtracted coupling matrices from {len(coupling_matrices)} layers")
for layer_idx, matrix in coupling_matrices.items():
    print(f"  Layer {layer_idx}: {matrix.shape} - range [{matrix.min():.4f}, {matrix.max():.4f}]")

In [ ]:
## 3. Process Connectivity Matrices for Network Analysis

def process_connectivity_for_network_analysis(coupling_matrices):
    """
    Process 4D coupling matrices into 2D coupling matrices suitable for network analysis.
    
    The AKOrN connectivity matrices are typically 4D: (out_channels, in_channels, kernel_h, kernel_w)
    We need to convert them to 2D matrices representing node-to-node coupling.
    """
    processed_matrices = {}
    
    for layer_idx, matrix in coupling_matrices.items():
        print(f"\nProcessing Layer {layer_idx}:")
        print(f"  Original shape: {matrix.shape}")
        
        if matrix.ndim == 4:
            # 4D conv weights: (out_ch, in_ch, h, w)
            out_ch, in_ch, h, w = matrix.shape
            
            # Method 1: Flatten spatial dimensions and reshape
            # This creates a coupling matrix between input and output channels
            matrix_2d = matrix.reshape(out_ch, in_ch * h * w)
            
            # Method 2: Average over spatial dimensions (alternative approach)
            matrix_spatial_avg = matrix.mean(axis=(2, 3))
            
            # Method 3: Frobenius norm over spatial dimensions
            matrix_frob = np.linalg.norm(matrix, axis=(2, 3))
            
            print(f"  Method 1 (flatten spatial): {matrix_2d.shape}")
            print(f"  Method 2 (spatial average): {matrix_spatial_avg.shape}")  
            print(f"  Method 3 (Frobenius norm): {matrix_frob.shape}")
            
            # For network analysis, we'll use the spatial average method
            # and then make it square by taking the correlation matrix if needed
            if matrix_spatial_avg.shape[0] == matrix_spatial_avg.shape[1]:
                # Already square
                processed_matrices[layer_idx] = matrix_spatial_avg
            else:
                # Make square by computing correlation between channels
                if matrix_spatial_avg.shape[0] < matrix_spatial_avg.shape[1]:
                    # More input channels than output channels
                    corr_matrix = np.corrcoef(matrix_spatial_avg)
                    processed_matrices[layer_idx] = corr_matrix
                else:
                    # More output channels than input channels  
                    corr_matrix = np.corrcoef(matrix_spatial_avg.T)
                    processed_matrices[layer_idx] = corr_matrix
                    
        elif matrix.ndim == 2:
            # Already 2D
            processed_matrices[layer_idx] = matrix
            
        else:
            print(f"  Unsupported matrix dimensionality: {matrix.ndim}")
            continue
            
        final_shape = processed_matrices[layer_idx].shape
        print(f"  Final processed shape: {final_shape}")
        print(f"  Is square: {final_shape[0] == final_shape[1]}")
        print(f"  Value range: [{processed_matrices[layer_idx].min():.4f}, {processed_matrices[layer_idx].max():.4f}]")
    
    return processed_matrices

# Process the coupling matrices
processed_coupling_matrices = process_connectivity_for_network_analysis(coupling_matrices)

In [ ]:
## 4. Compute Network Metrics for Each Layer

def analyze_layer_networks(processed_matrices):
    """Compute comprehensive network metrics for each layer."""
    layer_network_metrics = {}
    
    for layer_idx, K in processed_matrices.items():
        print(f"\n=== Network Analysis for Layer {layer_idx} ===")
        print(f"Coupling matrix shape: {K.shape}")
        
        try:
            # Compute all network metrics
            metrics = compute_all_metrics(K, directed=False, threshold=1e-6)
            layer_network_metrics[layer_idx] = metrics
            
            # Print key metrics
            print(f"\\nSpectral metrics:")
            print(f"  λ₂ (algebraic connectivity): {metrics['spectral']['lambda_2']:.6f}")
            print(f"  λ_N (largest eigenvalue): {metrics['spectral']['lambda_N']:.6f}")
            print(f"  Eigenratio (λ_N/λ₂): {metrics['spectral']['eigenratio']:.2f}")
            
            print(f"\\nStrength metrics:")
            strengths = metrics['strength']['strength']
            print(f"  Mean strength: {strengths.mean():.4f}")
            print(f"  Std strength: {strengths.std():.4f}")
            print(f"  Max strength: {strengths.max():.4f}")
            
            print(f"\\nCommunity metrics:")
            print(f"  Modularity Q: {metrics['community']['modularity']:.4f}")
            partition = metrics['community']['partition']
            n_communities = len(set(partition.values()))
            print(f"  Number of communities: {n_communities}")
            
            print(f"\\nPath metrics:")
            print(f"  Average shortest path: {metrics['path']['avg_shortest_path']:.4f}")
            print(f"  Diameter: {metrics['path']['diameter']:.4f}")
            
            # Core metrics
            coreness = metrics['kcore']['coreness']
            print(f"\\nCore metrics:")
            print(f"  Max k-core: {coreness.max()}")
            print(f"  Mean coreness: {coreness.mean():.2f}")
            
        except Exception as e:
            print(f"Error analyzing layer {layer_idx}: {e}")
            layer_network_metrics[layer_idx] = None
    
    return layer_network_metrics

# Analyze all layers
network_metrics = analyze_layer_networks(processed_coupling_matrices)

In [ ]:
## 5. Visualize Network Properties

def plot_network_comparison(network_metrics):
    """Create comprehensive plots comparing network properties across layers."""
    
    # Extract data for plotting
    layers = list(network_metrics.keys())
    valid_layers = [l for l in layers if network_metrics[l] is not None]
    
    if not valid_layers:
        print("No valid network metrics to plot")
        return
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Spectral properties
    lambda2_vals = [network_metrics[l]['spectral']['lambda_2'] for l in valid_layers]
    lambdaN_vals = [network_metrics[l]['spectral']['lambda_N'] for l in valid_layers]
    eigenratios = [network_metrics[l]['spectral']['eigenratio'] for l in valid_layers]
    
    axes[0,0].bar(range(len(valid_layers)), lambda2_vals, alpha=0.7, label='λ₂')
    axes[0,0].bar(range(len(valid_layers)), lambdaN_vals, alpha=0.7, label='λ_N')
    axes[0,0].set_xlabel('Layer')
    axes[0,0].set_ylabel('Eigenvalue')
    axes[0,0].set_title('Laplacian Eigenvalues')
    axes[0,0].set_xticks(range(len(valid_layers)))
    axes[0,0].set_xticklabels([f'L{l}' for l in valid_layers])
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Eigenratio
    axes[0,1].plot(range(len(valid_layers)), eigenratios, 'o-', linewidth=2, markersize=8)
    axes[0,1].set_xlabel('Layer')
    axes[0,1].set_ylabel('Eigenratio (λ_N/λ₂)')
    axes[0,1].set_title('Spectral Gap Ratio')
    axes[0,1].set_xticks(range(len(valid_layers)))
    axes[0,1].set_xticklabels([f'L{l}' for l in valid_layers])
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Strength distribution
    axes[0,2].set_title('Strength Distributions by Layer')
    for i, layer in enumerate(valid_layers):
        strengths = network_metrics[layer]['strength']['strength']
        axes[0,2].hist(strengths, bins=20, alpha=0.6, label=f'Layer {layer}')
    axes[0,2].set_xlabel('Node Strength')
    axes[0,2].set_ylabel('Count')
    axes[0,2].legend()
    axes[0,2].grid(True, alpha=0.3)
    
    # 4. Modularity
    modularities = [network_metrics[l]['community']['modularity'] for l in valid_layers]
    axes[1,0].bar(range(len(valid_layers)), modularities, alpha=0.7, color='green')
    axes[1,0].set_xlabel('Layer')
    axes[1,0].set_ylabel('Modularity Q')
    axes[1,0].set_title('Community Structure')
    axes[1,0].set_xticks(range(len(valid_layers)))
    axes[1,0].set_xticklabels([f'L{l}' for l in valid_layers])
    axes[1,0].grid(True, alpha=0.3)
    
    # 5. Path length metrics
    avg_paths = [network_metrics[l]['path']['avg_shortest_path'] for l in valid_layers]
    diameters = [network_metrics[l]['path']['diameter'] for l in valid_layers]
    
    x = range(len(valid_layers))
    axes[1,1].bar(x, avg_paths, alpha=0.7, label='Avg path length')
    axes[1,1].bar(x, diameters, alpha=0.7, label='Diameter')
    axes[1,1].set_xlabel('Layer')
    axes[1,1].set_ylabel('Path Length')
    axes[1,1].set_title('Path Length Metrics')
    axes[1,1].set_xticks(range(len(valid_layers)))
    axes[1,1].set_xticklabels([f'L{l}' for l in valid_layers])
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    # 6. K-core distribution
    axes[1,2].set_title('K-Core Distributions by Layer')
    for i, layer in enumerate(valid_layers):
        coreness = network_metrics[layer]['kcore']['coreness']
        unique_cores, counts = np.unique(coreness, return_counts=True)
        axes[1,2].bar(unique_cores + i*0.2, counts, width=0.2, alpha=0.7, label=f'Layer {layer}')
    axes[1,2].set_xlabel('K-Core Number')
    axes[1,2].set_ylabel('Count')
    axes[1,2].legend()
    axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create the comparison plots
plot_network_comparison(network_metrics)

In [ ]:
## 6. Network Visualization

def visualize_layer_networks(processed_matrices, network_metrics, max_nodes=50):
    """Visualize the actual network graphs for each layer."""
    
    valid_layers = [l for l in network_metrics.keys() if network_metrics[l] is not None]
    
    fig, axes = plt.subplots(1, len(valid_layers), figsize=(6*len(valid_layers), 6))
    if len(valid_layers) == 1:
        axes = [axes]
    
    for i, layer_idx in enumerate(valid_layers):
        K = processed_matrices[layer_idx]
        
        # Limit size for visualization
        if K.shape[0] > max_nodes:
            print(f"Layer {layer_idx}: Reducing from {K.shape[0]} to {max_nodes} nodes for visualization")
            # Take the top nodes by strength
            strengths = network_metrics[layer_idx]['strength']['strength']
            top_indices = np.argsort(strengths)[-max_nodes:]
            K_viz = K[np.ix_(top_indices, top_indices)]
        else:
            K_viz = K
        
        # Create NetworkX graph
        G = graph_from_K(K_viz, directed=False, threshold=np.percentile(np.abs(K_viz), 75))
        
        # Get community partition for coloring
        if K_viz.shape[0] <= max_nodes:
            try:
                community_data = community_metrics(K_viz, directed=False)
                partition = community_data['partition']
                colors = [partition.get(node, 0) for node in G.nodes()]
            except:
                colors = 'blue'
        else:
            colors = 'blue'
        
        # Layout
        if G.number_of_nodes() > 0:
            pos = nx.spring_layout(G, k=1/np.sqrt(G.number_of_nodes()), iterations=50)
            
            # Draw network
            nx.draw(G, pos, ax=axes[i], 
                   node_color=colors, 
                   node_size=50,
                   edge_color='gray',
                   alpha=0.7,
                   with_labels=False)
        
        axes[i].set_title(f'Layer {layer_idx} Network\\n({K_viz.shape[0]} nodes, {G.number_of_edges()} edges)')
    
    plt.tight_layout()
    plt.show()

# Visualize networks
visualize_layer_networks(processed_coupling_matrices, network_metrics)

In [ ]:
## 7. Comparative Analysis Across Sweep Models

def analyze_sweep_models(sweep_dirs, layer_idx=0):
    """Compare network metrics across different models from the parameter sweep."""
    
    sweep_network_data = {}
    
    print(f"Analyzing network properties across {len(sweep_dirs)} sweep models for layer {layer_idx}...")
    
    for sweep_dir in sweep_dirs:
        print(f"\\nProcessing {sweep_dir}...")
        
        # Load configuration
        config_path = results_dir / sweep_dir / "parameters.json"
        model_path = results_dir / sweep_dir / "my_akorn_cifar10_final.pth"
        
        if not config_path.exists() or not model_path.exists():
            print(f"  Skipping {sweep_dir}: missing files")
            continue
            
        try:
            # Load config
            with open(config_path, 'r') as f:
                config = json.load(f)
            
            # Create and load model
            sweep_model = MyAKOrN(
                n=config['n'],
                ch=config['ch'], 
                out_classes=config['num_classes'],
                L=config['L'],
                T=config['T'],
                J=config['J'],
                J_bias=config['J_bias'],
                ksizes=config['ksizes'],
                ro_ksize=config['ro_ksize'],
                ro_N=config['ro_N'],
                norm=config['norm'],
                c_norm=config['c_norm'],
                gamma=config['gamma'],
                use_omega=config['use_omega'],
                init_omg=config['init_omg'],
                global_omg=config['global_omg'],
                learn_omg=config['learn_omg'],
                ensemble=config['ensemble']
            ).to(device)
            
            checkpoint = torch.load(model_path, map_location=device)
            sweep_model.load_state_dict(checkpoint['model_state_dict'])
            sweep_model.eval()
            
            # Extract coupling matrix for specific layer
            coupling_matrices = extract_coupling_matrices(sweep_model, layer_idx=layer_idx)
            if layer_idx not in coupling_matrices:
                print(f"  Layer {layer_idx} not found in {sweep_dir}")
                continue
                
            # Process for network analysis
            processed_matrices = process_connectivity_for_network_analysis(coupling_matrices)
            if layer_idx not in processed_matrices:
                print(f"  Could not process layer {layer_idx} in {sweep_dir}")
                continue
            
            # Compute network metrics
            K = processed_matrices[layer_idx]
            metrics = compute_all_metrics(K, directed=False, threshold=1e-6)
            
            # Store results
            sweep_network_data[sweep_dir] = {
                'config': config,
                'metrics': metrics,
                'gamma': config['gamma'],
                'T': config['T']
            }
            
            print(f"  ✓ Analyzed {sweep_dir}: γ={config['gamma']}, T={config['T']}")
            
        except Exception as e:
            print(f"  ✗ Error with {sweep_dir}: {e}")
            continue
    
    print(f"\\nSuccessfully analyzed {len(sweep_network_data)} models")
    return sweep_network_data

# Analyze sweep models (this might take a while)
sweep_network_data = analyze_sweep_models(sweep_dirs[:5], layer_idx=0)  # Start with first 5 models

In [ ]:
## 8. Analyze Parameter Dependencies

def plot_parameter_dependencies(sweep_network_data):
    """Plot how network metrics depend on gamma and T parameters."""
    
    if len(sweep_network_data) < 2:
        print("Need at least 2 models for parameter dependency analysis")
        return
    
    # Extract data
    gammas = [data['gamma'] for data in sweep_network_data.values()]
    Ts = [data['T'] for data in sweep_network_data.values()]
    
    # Extract network metrics
    lambda2s = [data['metrics']['spectral']['lambda_2'] for data in sweep_network_data.values()]
    eigenratios = [data['metrics']['spectral']['eigenratio'] for data in sweep_network_data.values()]
    modularities = [data['metrics']['community']['modularity'] for data in sweep_network_data.values()]
    avg_paths = [data['metrics']['path']['avg_shortest_path'] for data in sweep_network_data.values()]
    
    # Create plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Eigenratio vs gamma
    axes[0,0].scatter(gammas, eigenratios, c=Ts, cmap='viridis', s=100, alpha=0.7)
    axes[0,0].set_xlabel('Gamma')
    axes[0,0].set_ylabel('Eigenratio (λ_N/λ₂)')
    axes[0,0].set_title('Eigenratio vs Gamma (colored by T)')
    axes[0,0].set_xscale('log')
    cbar1 = plt.colorbar(axes[0,0].collections[0], ax=axes[0,0])
    cbar1.set_label('T value')
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Modularity vs gamma
    axes[0,1].scatter(gammas, modularities, c=Ts, cmap='viridis', s=100, alpha=0.7)
    axes[0,1].set_xlabel('Gamma')
    axes[0,1].set_ylabel('Modularity Q')
    axes[0,1].set_title('Modularity vs Gamma (colored by T)')
    axes[0,1].set_xscale('log')
    cbar2 = plt.colorbar(axes[0,1].collections[0], ax=axes[0,1])
    cbar2.set_label('T value')
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Path length vs T
    axes[1,0].scatter(Ts, avg_paths, c=gammas, cmap='plasma', s=100, alpha=0.7)
    axes[1,0].set_xlabel('T')
    axes[1,0].set_ylabel('Average Path Length')
    axes[1,0].set_title('Path Length vs T (colored by γ)')
    cbar3 = plt.colorbar(axes[1,0].collections[0], ax=axes[1,0])
    cbar3.set_label('Gamma value')
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Algebraic connectivity vs gamma
    axes[1,1].scatter(gammas, lambda2s, c=Ts, cmap='viridis', s=100, alpha=0.7)
    axes[1,1].set_xlabel('Gamma')
    axes[1,1].set_ylabel('λ₂ (Algebraic Connectivity)')
    axes[1,1].set_title('Algebraic Connectivity vs Gamma (colored by T)')
    axes[1,1].set_xscale('log')
    cbar4 = plt.colorbar(axes[1,1].collections[0], ax=axes[1,1])
    cbar4.set_label('T value')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print correlation analysis
    print("\\n=== Parameter Correlation Analysis ===")
    
    # Convert to log scale for gamma
    log_gammas = np.log10(gammas)
    
    correlations = {
        'log(gamma) vs eigenratio': np.corrcoef(log_gammas, eigenratios)[0,1],
        'log(gamma) vs modularity': np.corrcoef(log_gammas, modularities)[0,1],
        'log(gamma) vs λ₂': np.corrcoef(log_gammas, lambda2s)[0,1],
        'T vs avg_path': np.corrcoef(Ts, avg_paths)[0,1],
        'T vs eigenratio': np.corrcoef(Ts, eigenratios)[0,1],
    }
    
    for metric, corr in correlations.items():
        print(f"{metric}: {corr:.3f}")

# Plot parameter dependencies
plot_parameter_dependencies(sweep_network_data)

In [ ]:
## 9. Summary and Interpretation

def generate_network_analysis_summary(network_metrics, sweep_network_data):
    """Generate a comprehensive summary of network analysis findings."""
    
    print("="*60)
    print("NETWORK ANALYSIS SUMMARY")
    print("="*60)
    
    print("\\n1. SINGLE MODEL LAYER ANALYSIS:")
    print("-" * 40)
    
    for layer_idx, metrics in network_metrics.items():
        if metrics is None:
            continue
            
        print(f"\\nLayer {layer_idx}:")
        print(f"  • Spectral properties:")
        print(f"    - λ₂ (algebraic connectivity): {metrics['spectral']['lambda_2']:.6f}")
        print(f"    - Eigenratio (λ_N/λ₂): {metrics['spectral']['eigenratio']:.2f}")
        
        print(f"  • Community structure:")
        print(f"    - Modularity Q: {metrics['community']['modularity']:.4f}")
        n_communities = len(set(metrics['community']['partition'].values()))
        print(f"    - Number of communities: {n_communities}")
        
        print(f"  • Path metrics:")
        print(f"    - Average path length: {metrics['path']['avg_shortest_path']:.4f}")
        print(f"    - Network diameter: {metrics['path']['diameter']:.1f}")
        
        strengths = metrics['strength']['strength']
        print(f"  • Strength distribution:")
        print(f"    - Mean: {strengths.mean():.4f}, Std: {strengths.std():.4f}")
        
        cores = metrics['kcore']['coreness']
        print(f"  • K-core structure:")
        print(f"    - Max k-core: {cores.max()}, Mean: {cores.mean():.2f}")
    
    if len(sweep_network_data) > 1:
        print("\\n\\n2. PARAMETER SWEEP ANALYSIS:")
        print("-" * 40)
        
        # Extract parameter ranges
        gammas = [data['gamma'] for data in sweep_network_data.values()]
        Ts = [data['T'] for data in sweep_network_data.values()]
        eigenratios = [data['metrics']['spectral']['eigenratio'] for data in sweep_network_data.values()]
        modularities = [data['metrics']['community']['modularity'] for data in sweep_network_data.values()]
        
        print(f"\\nParameter ranges analyzed:")
        print(f"  • Gamma: {min(gammas):.3f} - {max(gammas):.3f}")
        print(f"  • T: {min(Ts)} - {max(Ts)}")
        
        print(f"\\nNetwork metric ranges:")
        print(f"  • Eigenratio: {min(eigenratios):.2f} - {max(eigenratios):.2f}")
        print(f"  • Modularity: {min(modularities):.4f} - {max(modularities):.4f}")
        
        # Key findings
        print(f"\\nKey findings:")
        if len(set(gammas)) > 1:
            print(f"  • Gamma variation shows network structure dependency")
        if len(set(Ts)) > 1:
            print(f"  • T parameter affects temporal integration and connectivity")
        
    print("\\n\\n3. KURAMOTO NETWORK INTERPRETATION:")
    print("-" * 40)
    
    # Provide interpretation in context of Kuramoto oscillator networks
    for layer_idx, metrics in network_metrics.items():
        if metrics is None:
            continue
            
        lambda2 = metrics['spectral']['lambda_2']
        eigenratio = metrics['spectral']['eigenratio']
        modularity = metrics['community']['modularity']
        
        print(f"\\nLayer {layer_idx} - Kuramoto dynamics implications:")
        
        if lambda2 > 0.01:
            print(f"  • High algebraic connectivity (λ₂={lambda2:.4f}) → Strong synchronization potential")
        else:
            print(f"  • Low algebraic connectivity (λ₂={lambda2:.6f}) → Weak synchronization")
            
        if eigenratio < 10:
            print(f"  • Low eigenratio ({eigenratio:.2f}) → Good synchronization stability")
        else:
            print(f"  • High eigenratio ({eigenratio:.2f}) → Potential synchronization challenges")
            
        if modularity > 0.3:
            print(f"  • High modularity ({modularity:.3f}) → Clustered synchronization likely")
        else:
            print(f"  • Low modularity ({modularity:.3f}) → Uniform synchronization pattern")
    
    print("\\n" + "="*60)

# Generate comprehensive summary
generate_network_analysis_summary(network_metrics, sweep_network_data)